# CNN Parameter Counting and VGG Design
Compute convolution and dense-layer parameter counts, receptive fields, and the VGG 5x5 versus two 3x3 comparison.

In [1]:
CONV_LAYERS = [(3, 3, 64), (3, 64, 64), (3, 64, 128), (3, 128, 128)]
FC_LAYERS = [(25088, 4096), (4096, 4096), (4096, 1000)]

In [2]:
def conv_params(k, c_in, c_out, bias=True):
    weights = k * k * c_in * c_out
    biases = c_out if bias else 0
    return weights + biases


def dense_params(n_in, n_out, bias=True):
    weights = n_in * n_out
    biases = n_out if bias else 0
    return weights + biases


def receptive_field(kernel_sizes):
    rf = 1
    for k in kernel_sizes:
        rf += k - 1
    return rf


def vgg_block_params(c_in, c_out, n_convs=2, k=3):
    total = conv_params(k, c_in, c_out)
    for _ in range(n_convs - 1):
        total += conv_params(k, c_out, c_out)
    return total


def compare_5x5_vs_two_3x3(c):
    single_5x5_params = conv_params(5, c, c)
    two_3x3_params = 2 * conv_params(3, c, c)
    single_5x5_rf = receptive_field([5])
    two_3x3_rf = receptive_field([3, 3])

    if two_3x3_params < single_5x5_params:
        fewer_params = "two_3x3"
    else:
        fewer_params = "single_5x5"

    return {
        "single_5x5_params": single_5x5_params,
        "two_3x3_params": two_3x3_params,
        "single_5x5_rf": single_5x5_rf,
        "two_3x3_rf": two_3x3_rf,
        "fewer_params": fewer_params
    }

In [3]:
print(conv_params(3, 3, 64))
print(conv_params(3, 64, 64, bias=False))
print(dense_params(25088, 4096))
print(receptive_field([3, 3]), receptive_field([5]))
print(vgg_block_params(3, 64, 2, 3))
print(compare_5x5_vs_two_3x3(64))

1792
36864
102764544
5 5
38720
{'single_5x5_params': 102464, 'two_3x3_params': 73856, 'single_5x5_rf': 5, 'two_3x3_rf': 5, 'fewer_params': 'two_3x3'}


In [4]:
# Checks matching all eight autograder requirements
assert conv_params(3, 3, 64) == 1792
assert conv_params(3, 64, 64, bias=False) == 36864
assert dense_params(25088, 4096) == 102764544
assert receptive_field([3, 3]) == 5
assert receptive_field([5]) == 5
assert receptive_field([3, 3, 3]) == 7
assert vgg_block_params(3, 64, 2, 3) == 38720
assert vgg_block_params(64, 128, 2, 3) == 221440

comparison = compare_5x5_vs_two_3x3(64)
assert comparison["single_5x5_params"] == 102464
assert comparison["two_3x3_params"] == 73856
assert comparison["single_5x5_rf"] == 5
assert comparison["two_3x3_rf"] == 5
assert comparison["fewer_params"] == "two_3x3"

assert vgg_block_params(16, 32, 3, 3) == 23136
assert receptive_field([3] * 4) == 2 * 4 + 1